### Cell 1 — Initialize PRAGA repository and benchmark paths

In [51]:
from pathlib import Path
import shutil
import subprocess
import sys


WORK_ROOT = Path("/kaggle/working")

PRAGA_ROOT = (
    WORK_ROOT
    / "PRAGA"
)

PRAGA_OUTPUT_ROOT = (
    WORK_ROOT
    / "PRAGA_baseline"
)

DATA_ROOT = Path(
    "/kaggle/input/datasets/wuvdji/smgc-data"
)


print("=" * 100)
print("INITIALIZE PRAGA")
print("=" * 100)


# Fresh official repository
if PRAGA_ROOT.exists():

    shutil.rmtree(
        PRAGA_ROOT
    )


for name in list(sys.modules):

    if (
        name == "PRAGA"
        or name.startswith("PRAGA.")
        or name == "clustering_utils"
    ):

        del sys.modules[name]


subprocess.run(
    [
        "git",
        "clone",
        "https://github.com/Xubin-s-Lab/PRAGA.git",
        str(PRAGA_ROOT),
    ],
    check=True,
)


assert PRAGA_ROOT.exists()
assert DATA_ROOT.exists()


PRAGA_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


PRAGA_COMMIT = (
    subprocess.check_output(
        [
            "git",
            "-C",
            str(PRAGA_ROOT),
            "rev-parse",
            "HEAD",
        ],
        text=True,
    )
    .strip()
)


print(
    "\nPRAGA root :",
    PRAGA_ROOT
)

print(
    "Data root  :",
    DATA_ROOT
)

print(
    "Git commit :",
    PRAGA_COMMIT
)

print(
    "\nPASS: clean PRAGA repository ready."
)

INITIALIZE PRAGA


Cloning into '/kaggle/working/PRAGA'...



PRAGA root : /kaggle/working/PRAGA
Data root  : /kaggle/input/datasets/wuvdji/smgc-data
Git commit : 4adb11c96fc7ddad800fa1787eadcc8b91b42784

PASS: clean PRAGA repository ready.


### Cell 2 — Install PRAGA runtime dependencies

In [52]:
import sys
import subprocess


print("=" * 100)
print("INSTALL PRAGA DEPENDENCIES")
print("=" * 100)


subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--prefer-binary",
        "-q",
        "anndata==0.11.4",
        "scanpy==1.11.4",
        "scikit-learn==1.6.1",
        "scikit-misc",
    ],
    check=True,
)


# ============================================================
# R mclust
# ============================================================

mclust_ok = (
    subprocess.run(
        [
            "Rscript",
            "-e",
            (
                'quit(status='
                'ifelse(requireNamespace('
                '"mclust", quietly=TRUE),0,1))'
            ),
        ]
    ).returncode
    == 0
)


if not mclust_ok:

    subprocess.run(
        [
            "Rscript",
            "-e",
            (
                'install.packages('
                '"mclust", '
                'repos="https://cloud.r-project.org")'
            ),
        ],
        check=True,
    )


print(
    "\nPASS: dependencies ready."
)

INSTALL PRAGA DEPENDENCIES

PASS: dependencies ready.


### Cell 3 — Apply minimal PRAGA source compatibility patch

In [53]:
import sys
import re
import importlib

import numpy as np
import pandas as pd
import scipy
import scanpy as sc
import anndata
import sklearn
import torch


# ============================================================
# ONLY compatibility patch:
#
# official source:
#     cuda:1
#
# Kaggle single T4:
#     cuda:0
#
# Do NOT patch AMP, optimizer, loss or preprocessing here.
# ============================================================

patched_files = []


for path in PRAGA_ROOT.rglob("*.py"):

    text = path.read_text(
        encoding="utf-8"
    )

    if "cuda:1" in text:

        path.write_text(
            text.replace(
                "cuda:1",
                "cuda:0",
            ),
            encoding="utf-8",
        )

        patched_files.append(
            str(
                path.relative_to(
                    PRAGA_ROOT
                )
            )
        )


# ============================================================
# Clear stale modules
# ============================================================

for name in list(sys.modules):

    if (
        name == "PRAGA"
        or name.startswith("PRAGA.")
        or name == "clustering_utils"
    ):

        del sys.modules[name]


repo_root = str(
    PRAGA_ROOT
)

package_root = str(
    PRAGA_ROOT
    / "PRAGA"
)


sys.path = [
    p
    for p in sys.path
    if p not in {
        repo_root,
        package_root,
    }
]


sys.path.insert(
    0,
    package_root,
)

sys.path.insert(
    0,
    repo_root,
)


importlib.invalidate_caches()


# ============================================================
# Source audit
# ============================================================

main_text = (
    PRAGA_ROOT
    / "main.py"
).read_text(
    encoding="utf-8"
)


train_text = (
    PRAGA_ROOT
    / "PRAGA"
    / "Train_model.py"
).read_text(
    encoding="utf-8"
)


cluster_text = (
    PRAGA_ROOT
    / "PRAGA"
    / "optimal_clustering_HLN.py"
).read_text(
    encoding="utf-8"
)


assert (
    "adata_peaks_normalized.h5ad"
    in main_text
)

assert (
    "n_components=51"
    in main_text
)

assert (
    "self.epochs = 30"
    in train_text
)

assert (
    "self.epochs = 300"
    in train_text
)

assert (
    "Arg(init_K=14)"
    in re.sub(
        r"\s+",
        "",
        cluster_text,
    )
)


DEVICE = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 100)
print("PRAGA SOURCE / ENVIRONMENT AUDIT")
print("=" * 100)

print(
    "Python         :",
    sys.version.split()[0]
)

print(
    "PyTorch        :",
    torch.__version__
)

print(
    "CUDA available :",
    torch.cuda.is_available()
)


if torch.cuda.is_available():

    print(
        "GPU            :",
        torch.cuda.get_device_name(0)
    )


print(
    "NumPy          :",
    np.__version__
)

print(
    "SciPy          :",
    scipy.__version__
)

print(
    "Scanpy         :",
    sc.__version__
)

print(
    "AnnData        :",
    anndata.__version__
)

print(
    "sklearn        :",
    sklearn.__version__
)

print(
    "Device         :",
    DEVICE
)

print(
    "Patched files  :",
    patched_files
)


print(
    "\nPASS: only cuda:1 -> cuda:0 patched."
)

print(
    "PASS: no training or preprocessing patch applied."
)

PRAGA SOURCE / ENVIRONMENT AUDIT
Python         : 3.12.13
PyTorch        : 2.10.0+cu128
CUDA available : True
GPU            : Tesla T4
NumPy          : 2.0.2
SciPy          : 1.16.3
Scanpy         : 1.11.4
AnnData        : 0.11.4
sklearn        : 1.6.1
Device         : cuda:0
Patched files  : ['main.py', 'PRAGA/optimal_clustering_HLN.py', 'PRAGA/optimal_clustering.py']

PASS: only cuda:1 -> cuda:0 patched.
PASS: no training or preprocessing patch applied.


### Cell 4 — Define the five benchmark datasets and PRAGA settings

In [54]:
DATASET_ORDER = [
    "HLN-A1",
    "HLN-D1",
    "E18.5",
    "S2-E15",
    "S2-E18",
]


DATASET_SPECS = {

    "HLN-A1": {

        "rna":
            DATA_ROOT
            / "Human_Lymph_Nodes/A1/adata_RNA.h5ad",

        "second":
            DATA_ROOT
            / "Human_Lymph_Nodes/A1/adata_ADT.h5ad",

        "second_type":
            "ADT",

        "K":
            10,

        "n_spots":
            3484,

        "init_k":
            10,

        "KNN_k":
            20,

        "RNA_weight":
            5,

        "ADT_weight":
            5,

        "epochs":
            30,
    },


    "HLN-D1": {

        "rna":
            DATA_ROOT
            / "Human_Lymph_Nodes/D1/adata_RNA.h5ad",

        "second":
            DATA_ROOT
            / "Human_Lymph_Nodes/D1/adata_ADT.h5ad",

        "second_type":
            "ADT",

        "K":
            11,

        "n_spots":
            3359,

        "init_k":
            11,

        "KNN_k":
            20,

        "RNA_weight":
            5,

        "ADT_weight":
            5,

        "epochs":
            30,
    },


    "E18.5": {

        "rna":
            DATA_ROOT
            / "E18.5_mouse_brain/adata_RNA.h5ad",

        "second":
            DATA_ROOT
            / "E18.5_mouse_brain/adata_ATAC.h5ad",

        "second_type":
            "ATAC",

        "K":
            14,

        "n_spots":
            2129,

        "init_k":
            14,

        "KNN_k":
            20,

        "RNA_weight":
            1,

        "ADT_weight":
            10,

        "epochs":
            300,
    },


    "S2-E15": {

        "rna":
            DATA_ROOT
            / "Mouse_Embryos_S2/E15/adata_RNA.h5ad",

        "second":
            DATA_ROOT
            / "Mouse_Embryos_S2/E15/adata_ATAC.h5ad",

        "second_type":
            "ATAC",

        "K":
            15,

        "n_spots":
            1939,

        "init_k":
            14,

        "KNN_k":
            20,

        "RNA_weight":
            1,

        "ADT_weight":
            10,

        "epochs":
            300,
    },


    "S2-E18": {

        "rna":
            DATA_ROOT
            / "Mouse_Embryos_S2/E18/adata_RNA.h5ad",

        "second":
            DATA_ROOT
            / "Mouse_Embryos_S2/E18/adata_ATAC.h5ad",

        "second_type":
            "ATAC",

        "K":
            16,

        "n_spots":
            2248,

        "init_k":
            14,

        "KNN_k":
            20,

        "RNA_weight":
            1,

        "ADT_weight":
            10,

        "epochs":
            300,
    },
}


print(
    "PASS: five benchmark datasets configured."
)

PASS: five benchmark datasets configured.


### Cell 5 — Audit benchmark files, spot counts and annotations

In [55]:
def load_clean_dataset(
    dataset_name,
):

    spec = (
        DATASET_SPECS[
            dataset_name
        ]
    )


    assert spec["rna"].exists(), (
        spec["rna"]
    )

    assert spec["second"].exists(), (
        spec["second"]
    )


    rna = sc.read_h5ad(
        spec["rna"]
    )

    second = sc.read_h5ad(
        spec["second"]
    )


    rna.var_names_make_unique()
    second.var_names_make_unique()


    assert (
        rna.n_obs
        ==
        spec["n_spots"]
    )

    assert (
        second.n_obs
        ==
        spec["n_spots"]
    )


    assert np.array_equal(
        np.asarray(
            rna.obs_names
        ),
        np.asarray(
            second.obs_names
        ),
    )


    # ========================================================
    # E18.5 metadata adapter
    # Same benchmark convention used in previous notebooks
    # ========================================================

    if (
        dataset_name
        == "E18.5"
    ):

        coords = np.column_stack(
            [
                np.asarray(
                    rna.obs[
                        "array_col"
                    ],
                    dtype=np.float64,
                ),

                np.asarray(
                    rna.obs[
                        "array_row"
                    ],
                    dtype=np.float64,
                ),
            ]
        )


        labels = (
            rna.obs[
                "Combined_Clusters_annotation"
            ]
            .astype(str)
            .to_numpy()
        )


        rna.obsm[
            "spatial"
        ] = coords.copy()

        second.obsm[
            "spatial"
        ] = coords.copy()


        rna.obs[
            "Spatial_Label"
        ] = labels.copy()

        second.obs[
            "Spatial_Label"
        ] = labels.copy()


    # ========================================================
    # Audit
    # ========================================================

    assert (
        "Spatial_Label"
        in rna.obs.columns
    )

    assert (
        "spatial"
        in rna.obsm
    )

    assert (
        rna.obs[
            "Spatial_Label"
        ].nunique()
        ==
        spec["K"]
    )


    return (
        rna,
        second,
        spec,
    )


print("=" * 110)
print("FIVE-DATASET BENCHMARK AUDIT")
print("=" * 110)


for dataset_name in DATASET_ORDER:

    (
        rna,
        second,
        spec,
    ) = load_clean_dataset(
        dataset_name
    )


    print(
        f"\n{dataset_name}"
    )

    print(
        "  RNA shape        :",
        rna.shape
    )

    print(
        f"  {spec['second_type']} shape"
        f"{' ' * max(0, 7-len(spec['second_type']))}:",
        second.shape
    )

    print(
        "  spots            :",
        rna.n_obs
    )

    print(
        "  target K         :",
        spec["K"]
    )

    print(
        "  annotation K     :",
        rna.obs[
            "Spatial_Label"
        ].nunique()
    )

    print(
        "  spatial shape    :",
        np.asarray(
            rna.obsm[
                "spatial"
            ]
        ).shape
    )


print(
    "\nPASS: five benchmark datasets verified."
)

FIVE-DATASET BENCHMARK AUDIT


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



HLN-A1
  RNA shape        : (3484, 18085)
  ADT shape    : (3484, 31)
  spots            : 3484
  target K         : 10
  annotation K     : 10
  spatial shape    : (3484, 2)


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



HLN-D1
  RNA shape        : (3359, 18085)
  ADT shape    : (3359, 31)
  spots            : 3359
  target K         : 11
  annotation K     : 11
  spatial shape    : (3359, 2)

E18.5
  RNA shape        : (2129, 32285)
  ATAC shape   : (2129, 161461)
  spots            : 2129
  target K         : 14
  annotation K     : 14
  spatial shape    : (2129, 2)


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



S2-E15
  RNA shape        : (1939, 32285)
  ATAC shape   : (1939, 100329)
  spots            : 1939
  target K         : 15
  annotation K     : 15
  spatial shape    : (1939, 2)


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



S2-E18
  RNA shape        : (2248, 32285)
  ATAC shape   : (2248, 94941)
  spots            : 2248
  target K         : 16
  annotation K     : 16
  spatial shape    : (2248, 2)

PASS: five benchmark datasets verified.


### Cell 6 — Audit RNA+ATAC peak-frequency filtering before LSI

In [56]:
import scipy.sparse as sp


print("=" * 110)
print("RNA+ATAC PEAK-FREQUENCY AUDIT")
print("=" * 110)


def peak_detection_counts(
    X,
):

    if sp.issparse(
        X
    ):

        # Number of spots in which each peak is non-zero
        detected = np.asarray(
            (
                X > 0
            ).sum(
                axis=0
            )
        ).ravel()

        total = np.asarray(
            X.sum(
                axis=0
            )
        ).ravel()

    else:

        X = np.asarray(
            X
        )

        detected = (
            X > 0
        ).sum(
            axis=0
        )

        total = X.sum(
            axis=0
        )


    return (
        detected,
        total,
    )


audit_rows = []


for dataset_name in [
    "E18.5",
    "S2-E15",
    "S2-E18",
]:

    spec = (
        DATASET_SPECS[
            dataset_name
        ]
    )


    atac = sc.read_h5ad(
        spec[
            "second"
        ]
    )


    detected, total = (
        peak_detection_counts(
            atac.X
        )
    )


    n_peaks = (
        atac.n_vars
    )


    zero_total = int(
        (
            total == 0
        ).sum()
    )


    at_least_1 = int(
        (
            detected >= 1
        ).sum()
    )

    at_least_2 = int(
        (
            detected >= 2
        ).sum()
    )

    at_least_5 = int(
        (
            detected >= 5
        ).sum()
    )

    at_least_10 = int(
        (
            detected >= 10
        ).sum()
    )

    at_least_20 = int(
        (
            detected >= 20
        ).sum()
    )


    print(
        "\n"
        + "-" * 110
    )

    print(
        dataset_name
    )

    print(
        "-" * 110
    )


    print(
        "spots                  :",
        atac.n_obs
    )

    print(
        "raw peaks              :",
        n_peaks
    )

    print(
        "zero-total peaks       :",
        zero_total
    )

    print(
        "detected >= 1 spot     :",
        at_least_1
    )

    print(
        "detected >= 2 spots    :",
        at_least_2
    )

    print(
        "detected >= 5 spots    :",
        at_least_5
    )

    print(
        "detected >= 10 spots   :",
        at_least_10
    )

    print(
        "detected >= 20 spots   :",
        at_least_20
    )


    audit_rows.append(
        {
            "dataset":
                dataset_name,

            "spots":
                atac.n_obs,

            "raw_peaks":
                n_peaks,

            "zero_total_peaks":
                zero_total,

            "peaks_ge_1":
                at_least_1,

            "peaks_ge_2":
                at_least_2,

            "peaks_ge_5":
                at_least_5,

            "peaks_ge_10":
                at_least_10,

            "peaks_ge_20":
                at_least_20,
        }
    )


peak_audit = pd.DataFrame(
    audit_rows
)


print(
    "\n"
    + "=" * 110
)

print(
    "PEAK-FREQUENCY SUMMARY"
)

print(
    "=" * 110
)


display(
    peak_audit
)


print(
    "\nAUDIT ONLY — no ATAC matrix was modified."
)

print(
    "Do NOT start PRAGA training yet."
)

RNA+ATAC PEAK-FREQUENCY AUDIT

--------------------------------------------------------------------------------------------------------------
E18.5
--------------------------------------------------------------------------------------------------------------
spots                  : 2129
raw peaks              : 161461
zero-total peaks       : 4
detected >= 1 spot     : 161457
detected >= 2 spots    : 161450
detected >= 5 spots    : 161249
detected >= 10 spots   : 158701
detected >= 20 spots   : 139203

--------------------------------------------------------------------------------------------------------------
S2-E15
--------------------------------------------------------------------------------------------------------------
spots                  : 1939
raw peaks              : 100329
zero-total peaks       : 0
detected >= 1 spot     : 100329
detected >= 2 spots    : 100329
detected >= 5 spots    : 100328
detected >= 10 spots   : 100060
detected >= 20 spots   : 95309

-------------

,dataset,spots,raw_peaks,zero_total_peaks,peaks_ge_1,peaks_ge_2,peaks_ge_5,peaks_ge_10,peaks_ge_20
0,E18.5,2129,161461,4,161457,161450,161249,158701,139203
1,S2-E15,1939,100329,0,100329,100329,100328,100060,95309
2,S2-E18,2248,94941,0,94941,94941,94941,94833,91828



AUDIT ONLY — no ATAC matrix was modified.
Do NOT start PRAGA training yet.


### Cell 7 — Define frozen PRAGA preprocessing for all five datasets

In [57]:
from types import SimpleNamespace
import scipy.sparse as sp

from PRAGA.preprocess import (
    fix_seed,
    clr_normalize_each_cell,
    pca,
    lsi,
    construct_neighbor_graph,
)

from PRAGA.Train_model import Train
from PRAGA.utils import clustering


def make_praga_args(spec):

    return SimpleNamespace(
        init_k=int(spec["init_k"]),
        KNN_k=int(spec["KNN_k"]),
        alpha=0.9,
        cl_weight=1.0,
        RNA_weight=float(spec["RNA_weight"]),
        ADT_weight=float(spec["ADT_weight"]),
        tau=2.0,
    )


def remove_zero_total_peaks_only(
    atac,
    dataset_name,
):

    n_spots_before = atac.n_obs
    n_peaks_before = atac.n_vars

    peak_sum = np.asarray(
        atac.X.sum(axis=0)
    ).ravel()

    assert np.isfinite(
        peak_sum
    ).all()

    keep = (
        peak_sum > 0
    )

    n_removed = int(
        (~keep).sum()
    )

    atac = (
        atac[:, keep]
        .copy()
    )

    assert (
        atac.n_obs
        ==
        n_spots_before
    )

    spot_sum = np.asarray(
        atac.X.sum(axis=1)
    ).ravel()

    zero_spots = int(
        (spot_sum <= 0).sum()
    )

    if zero_spots > 0:

        raise RuntimeError(
            f"{dataset_name}: "
            f"{zero_spots} zero-total ATAC spots."
        )

    print(
        f"{dataset_name}: "
        f"ATAC {n_peaks_before} -> {atac.n_vars} peaks "
        f"(removed {n_removed} zero-total peaks)"
    )

    return (
        atac,
        {
            "n_peaks_before":
                int(n_peaks_before),

            "zero_total_peaks_removed":
                int(n_removed),

            "n_peaks_after":
                int(atac.n_vars),

            "zero_total_spots":
                int(zero_spots),
        },
    )


def standardize_lsi_columns(
    X,
    dataset_name,
):

    X = np.asarray(
        X,
        dtype=np.float64,
    )

    assert np.isfinite(
        X
    ).all()

    mean = X.mean(
        axis=0,
        keepdims=True,
    )

    std = X.std(
        axis=0,
        ddof=1,
        keepdims=True,
    )

    if np.any(
        std <= 0
    ):

        raise RuntimeError(
            f"{dataset_name}: "
            "non-positive LSI column std."
        )

    X = (
        X - mean
    ) / std

    assert np.isfinite(
        X
    ).all()

    print(
        f"{dataset_name}: "
        f"scaled LSI shape = {X.shape}"
    )

    print(
        f"{dataset_name}: "
        f"max |column mean| = "
        f"{np.abs(X.mean(axis=0)).max():.3e}"
    )

    print(
        f"{dataset_name}: "
        f"sample std range = "
        f"[{X.std(axis=0, ddof=1).min():.6f}, "
        f"{X.std(axis=0, ddof=1).max():.6f}]"
    )

    return X


def preprocess_praga_frozen(
    dataset_name,
):

    (
        rna,
        second,
        spec,
    ) = load_clean_dataset(
        dataset_name
    )

    args = make_praga_args(
        spec
    )

    # ========================================================
    # RNA + ADT
    # ========================================================

    if (
        spec["second_type"]
        == "ADT"
    ):

        data_type = "10x"

        sc.pp.filter_genes(
            rna,
            min_cells=10,
        )

        sc.pp.highly_variable_genes(
            rna,
            flavor="seurat_v3",
            n_top_genes=3000,
        )

        sc.pp.normalize_total(
            rna,
            target_sum=1e4,
        )

        sc.pp.log1p(
            rna
        )

        sc.pp.scale(
            rna
        )

        rna_high = (
            rna[
                :,
                rna.var[
                    "highly_variable"
                ],
            ]
            .copy()
        )

        n_comps = (
            second.n_vars - 1
        )

        rna.obsm[
            "feat"
        ] = pca(
            rna_high,
            n_comps=n_comps,
        )

        if sp.issparse(
            second.X
        ):

            second.X = (
                second.X.toarray()
            )

        second = (
            clr_normalize_each_cell(
                second
            )
        )

        sc.pp.scale(
            second
        )

        second.obsm[
            "feat"
        ] = pca(
            second,
            n_comps=n_comps,
        )

        audit = {
            "modality":
                "RNA+ADT",

            "compatibility_preprocessing":
                "none",

            "spot_filtering":
                False,

            "extra_LSI_column_scaling":
                False,
        }


    # ========================================================
    # RNA + ATAC
    # ========================================================

    else:

        data_type = (
            "Spatial-epigenome-transcriptome"
        )

        benchmark_spots = (
            rna.obs_names.copy()
        )

        # ----------------------------------------------------
        # RNA
        # ----------------------------------------------------

        sc.pp.filter_genes(
            rna,
            min_cells=10,
        )

        # Keep fixed benchmark spot population.
        # Do NOT call filter_cells(min_genes=200).

        sc.pp.highly_variable_genes(
            rna,
            flavor="seurat_v3",
            n_top_genes=3000,
        )

        sc.pp.normalize_total(
            rna,
            target_sum=1e4,
        )

        sc.pp.log1p(
            rna
        )

        sc.pp.scale(
            rna
        )

        rna_high = (
            rna[
                :,
                rna.var[
                    "highly_variable"
                ],
            ]
            .copy()
        )

        rna.obsm[
            "feat"
        ] = pca(
            rna_high,
            n_comps=50,
        )


        # ----------------------------------------------------
        # ATAC
        # ----------------------------------------------------

        second = (
            second[
                benchmark_spots
            ]
            .copy()
        )

        assert np.array_equal(
            np.asarray(
                benchmark_spots
            ),
            np.asarray(
                second.obs_names
            ),
        )

        (
            second,
            atac_audit,
        ) = remove_zero_total_peaks_only(
            second,
            dataset_name,
        )

        # Keep PRAGA fallback LSI itself unchanged.
        sc.pp.highly_variable_genes(
            second,
            flavor="seurat_v3",
            n_top_genes=3000,
        )

        lsi(
            second,
            use_highly_variable=False,
            n_components=51,
        )

        raw_lsi = np.asarray(
            second.obsm[
                "X_lsi"
            ],
            dtype=np.float64,
        )

        assert raw_lsi.shape == (
            spec["n_spots"],
            50,
        )

        assert np.isfinite(
            raw_lsi
        ).all()

        # Fixed compatibility transformation.
        scaled_lsi = (
            standardize_lsi_columns(
                raw_lsi,
                dataset_name,
            )
        )

        second.obsm[
            "X_lsi"
        ] = scaled_lsi.copy()

        second.obsm[
            "feat"
        ] = scaled_lsi.copy()

        audit = {
            **atac_audit,

            "modality":
                "RNA+ATAC",

            "compatibility_preprocessing":
                "remove_zero_total_peaks"
                "+column_standardize_LSI",

            "spot_filtering":
                False,

            "LSI_use_highly_variable":
                False,

            "LSI_components":
                50,

            "extra_LSI_column_scaling":
                True,

            "LSI_scaling":
                "column_zero_mean_"
                "unit_sample_std_ddof1",
        }


    # ========================================================
    # Final feature audit
    # ========================================================

    assert (
        rna.n_obs
        ==
        spec["n_spots"]
    )

    assert (
        second.n_obs
        ==
        spec["n_spots"]
    )

    assert np.array_equal(
        np.asarray(
            rna.obs_names
        ),
        np.asarray(
            second.obs_names
        ),
    )

    feat1 = np.asarray(
        rna.obsm[
            "feat"
        ]
    )

    feat2 = np.asarray(
        second.obsm[
            "feat"
        ]
    )

    assert np.isfinite(
        feat1
    ).all()

    assert np.isfinite(
        feat2
    ).all()

    print(
        f"{dataset_name}: "
        f"RNA feat = {feat1.shape}"
    )

    print(
        f"{dataset_name}: "
        f"{spec['second_type']} feat = {feat2.shape}"
    )

    print(
        f"{dataset_name}: "
        f"KNN_k={spec['KNN_k']}, "
        f"weights="
        f"[{spec['RNA_weight']}, "
        f"{spec['ADT_weight']}], "
        f"init_k={spec['init_k']}"
    )

    data = construct_neighbor_graph(
        rna,
        second,
        datatype=data_type,
        Arg=args,
    )

    return (
        rna,
        second,
        data,
        args,
        spec,
        data_type,
        audit,
    )


print(
    "PASS: frozen five-dataset PRAGA preprocessing defined."
)

PASS: frozen five-dataset PRAGA preprocessing defined.


### Cell 8 — Stable PCA20 + mclust EEE bridge with explicit column names

In [58]:
import numpy as np
import subprocess
import tempfile
from pathlib import Path

from PRAGA.preprocess import pca

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)


def mclust_eee_labels(
    X,
    n_clusters,
    random_seed=2020,
):
    """
    Robust bridge:

        NumPy PCA matrix
        -> TSV
        -> Rscript
        -> mclust::Mclust
        -> labels

    Clustering protocol remains:
        G = target K
        modelNames = "EEE"
        seed = 2020
    """

    X = np.asarray(
        X,
        dtype=np.float64,
    )

    assert X.ndim == 2
    assert np.isfinite(X).all()

    n, d = X.shape

    with tempfile.TemporaryDirectory() as tmpdir:

        tmpdir = Path(tmpdir)

        input_path = (
            tmpdir
            / "pca_matrix.tsv"
        )

        output_path = (
            tmpdir
            / "mclust_labels.txt"
        )

        script_path = (
            tmpdir
            / "run_mclust.R"
        )


        # ====================================================
        # Save PCA matrix
        # ====================================================

        np.savetxt(
            input_path,
            X,
            delimiter="\t",
            fmt="%.17g",
        )


        # ====================================================
        # R script
        # ====================================================

        r_script = r'''
suppressPackageStartupMessages(
    library(mclust)
)

args <- commandArgs(
    trailingOnly = TRUE
)

input_file  <- args[1]
output_file <- args[2]
seed        <- as.integer(args[3])
G           <- as.integer(args[4])


X <- as.matrix(
    read.table(
        input_file,
        header = FALSE,
        sep = "\t",
        check.names = FALSE,
        stringsAsFactors = FALSE
    )
)

storage.mode(X) <- "double"


# Explicit names entirely inside R.
colnames(X) <- paste0(
    "PC",
    seq_len(
        ncol(X)
    )
)


if (
    any(
        !is.finite(X)
    )
) {
    stop(
        "Non-finite values in PCA matrix."
    )
}


set.seed(
    seed
)


fit <- Mclust(
    X,
    G = G,
    modelNames = "EEE"
)


labels <- fit$classification


if (
    length(labels)
    !=
    nrow(X)
) {
    stop(
        "Classification length mismatch."
    )
}


write.table(
    labels,
    file = output_file,
    row.names = FALSE,
    col.names = FALSE,
    quote = FALSE
)
'''

        script_path.write_text(
            r_script,
            encoding="utf-8",
        )


        # ====================================================
        # Run mclust in an independent R process
        # ====================================================

        proc = subprocess.run(
            [
                "Rscript",
                str(script_path),
                str(input_path),
                str(output_path),
                str(int(random_seed)),
                str(int(n_clusters)),
            ],
            text=True,
            capture_output=True,
        )


        if proc.returncode != 0:

            print(
                "\nR stdout:"
            )

            print(
                proc.stdout
            )

            print(
                "\nR stderr:"
            )

            print(
                proc.stderr
            )

            raise RuntimeError(
                "External R mclust failed."
            )


        assert output_path.exists()


        labels = np.loadtxt(
            output_path,
            dtype=np.int64,
        )


    labels = np.asarray(
        labels,
        dtype=np.int64,
    ).reshape(-1)


    assert labels.shape == (
        n,
    )

    assert np.isfinite(
        labels
    ).all()


    actual_k = int(
        len(
            np.unique(
                labels
            )
        )
    )


    if (
        actual_k
        !=
        int(n_clusters)
    ):

        raise RuntimeError(
            f"mclust returned "
            f"{actual_k} clusters; "
            f"expected {n_clusters}."
        )


    return labels


def cluster_praga_embedding(
    embedding,
    template_adata,
    target_k,
):

    embedding = np.asarray(
        embedding,
        dtype=np.float64,
    )

    assert embedding.ndim == 2

    assert np.isfinite(
        embedding
    ).all()


    # ========================================================
    # PRAGA embedding -> PCA20
    # ========================================================

    adata = (
        template_adata.copy()
    )

    adata.obsm[
        "PRAGA"
    ] = embedding.copy()


    embedding_pca = pca(
        adata,
        use_reps="PRAGA",
        n_comps=20,
    )

    embedding_pca = np.asarray(
        embedding_pca,
        dtype=np.float64,
    )


    assert embedding_pca.shape == (
        embedding.shape[0],
        20,
    )

    assert np.isfinite(
        embedding_pca
    ).all()


    print(
        "PCA20 shape :",
        embedding_pca.shape
    )

    print(
        "PCA20 finite:",
        np.isfinite(
            embedding_pca
        ).all()
    )


    # ========================================================
    # PCA20 -> mclust EEE
    # ========================================================

    pred = mclust_eee_labels(
        embedding_pca,
        n_clusters=int(
            target_k
        ),
        random_seed=2020,
    )


    print(
        "mclust K    :",
        len(
            np.unique(
                pred
            )
        )
    )


    return pred


def evaluate_praga(
    gt,
    pred,
):

    gt = np.asarray(
        gt
    ).astype(str)

    pred = np.asarray(
        pred
    ).astype(str)


    ari = adjusted_rand_score(
        gt,
        pred,
    )

    nmi = normalized_mutual_info_score(
        gt,
        pred,
        average_method="max",
    )


    return (
        float(ari),
        float(nmi),
    )


print(
    "PASS: external-Rscript PCA20 + mclust EEE bridge defined."
)

PASS: external-Rscript PCA20 + mclust EEE bridge defined.


### Cell 8.1 — Verify the external Rscript mclust bridge

In [59]:
rng = np.random.default_rng(
    2020
)


X_test = np.vstack(
    [
        rng.normal(
            loc=-3.0,
            scale=0.4,
            size=(40, 20),
        ),

        rng.normal(
            loc=0.0,
            scale=0.4,
            size=(40, 20),
        ),

        rng.normal(
            loc=3.0,
            scale=0.4,
            size=(40, 20),
        ),
    ]
)


test_labels = mclust_eee_labels(
    X_test,
    n_clusters=3,
    random_seed=2020,
)


print(
    "Input shape   :",
    X_test.shape
)

print(
    "Labels shape  :",
    test_labels.shape
)

print(
    "Predicted K   :",
    len(
        np.unique(
            test_labels
        )
    )
)

print(
    "Unique labels :",
    np.unique(
        test_labels
    )
)


assert (
    test_labels.shape
    ==
    (120,)
)

assert (
    len(
        np.unique(
            test_labels
        )
    )
    ==
    3
)


print(
    "\nPASS: external Rscript mclust bridge works."
)

Input shape   : (120, 20)
Labels shape  : (120,)
Predicted K   : 3
Unique labels : [1 2 3]

PASS: external Rscript mclust bridge works.


### Cell 9 — Define single-run PRAGA execution and evidence saving

In [60]:
import gc
import json
import time
from pathlib import Path


FORMAL_ROOT = (
    PRAGA_OUTPUT_ROOT
    / "formal"
)

FORMAL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


def dataset_tag(
    dataset_name,
):

    return (
        dataset_name
        .replace("-", "")
        .replace(".", "")
    )


def run_praga_once(
    dataset_name,
    seed,
    output_root=FORMAL_ROOT,
    overwrite=False,
):

    spec = (
        DATASET_SPECS[
            dataset_name
        ]
    )

    tag = dataset_tag(
        dataset_name
    )

    run_dir = (
        Path(output_root)
        / f"{tag}_seed{seed}"
    )

    metrics_path = (
        run_dir
        / "metrics.json"
    )


    required_files = [
        run_dir / "embedding.npy",
        run_dir / "pred_labels.npy",
        run_dir / "gt_labels.npy",
        run_dir / "coords.npy",
        run_dir / "spot_ids.npy",
        run_dir / "metrics.json",
        run_dir / "preprocessing_audit.json",
    ]


    # ========================================================
    # Resume
    # ========================================================

    if (
        not overwrite
        and all(
            p.exists()
            for p in required_files
        )
    ):

        with open(
            metrics_path,
            "r",
            encoding="utf-8",
        ) as f:

            result = json.load(
                f
            )

        print(
            f"SKIP: {dataset_name} seed={seed} "
            f"already complete."
        )

        return result


    run_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    print(
        "\n"
        + "=" * 110
    )

    print(
        f"PRAGA RUN | "
        f"{dataset_name} | "
        f"seed={seed}"
    )

    print(
        "=" * 110
    )


    start = time.time()


    # ========================================================
    # Seed
    # ========================================================

    fix_seed(
        int(seed)
    )


    # ========================================================
    # Preprocessing
    # ========================================================

    (
        rna,
        second,
        data,
        args,
        spec,
        data_type,
        preprocessing_audit,
    ) = preprocess_praga_frozen(
        dataset_name
    )


    # ========================================================
    # PRAGA training
    # ========================================================

    model = Train(
        data,
        datatype=data_type,
        device=DEVICE,
        random_seed=int(seed),
        Arg=args,
    )


    assert (
        int(model.epochs)
        ==
        int(spec["epochs"])
    )


    output = (
        model.train()
    )


    embedding = np.asarray(
        output[
            "PRAGA"
        ],
        dtype=np.float64,
    )


    assert embedding.shape == (
        spec["n_spots"],
        64,
    )

    assert np.isfinite(
        embedding
    ).all()


    # ========================================================
    # Final clustering
    # ========================================================

    pred = (
        cluster_praga_embedding(
            embedding,
            rna,
            spec["K"],
        )
    )


    gt = (
        rna.obs[
            "Spatial_Label"
        ]
        .astype(str)
        .to_numpy()
    )


    coords = np.asarray(
        rna.obsm[
            "spatial"
        ],
        dtype=np.float64,
    )


    spot_ids = np.asarray(
        rna.obs_names.astype(str)
    )


    assert len(
        np.unique(
            pred
        )
    ) == spec["K"]


    ari, nmi = evaluate_praga(
        gt,
        pred,
    )


    runtime_sec = float(
        time.time()
        -
        start
    )


    # ========================================================
    # Save evidence
    # ========================================================

    np.save(
        run_dir
        / "embedding.npy",
        embedding,
    )

    np.save(
        run_dir
        / "pred_labels.npy",
        pred,
    )

    np.save(
        run_dir
        / "gt_labels.npy",
        gt,
    )

    np.save(
        run_dir
        / "coords.npy",
        coords,
    )

    np.save(
        run_dir
        / "spot_ids.npy",
        spot_ids,
    )


    with open(
        run_dir
        / "preprocessing_audit.json",
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            preprocessing_audit,
            f,
            indent=2,
        )


    result = {
        "method":
            "PRAGA",

        "dataset":
            dataset_name,

        "seed":
            int(seed),

        "n_spots":
            int(spec["n_spots"]),

        "target_K":
            int(spec["K"]),

        "predicted_K":
            int(
                len(
                    np.unique(
                        pred
                    )
                )
            ),

        "training_init_K":
            int(
                spec[
                    "init_k"
                ]
            ),

        "epochs":
            int(
                spec[
                    "epochs"
                ]
            ),

        "KNN_k":
            int(
                spec[
                    "KNN_k"
                ]
            ),

        "RNA_weight":
            float(
                spec[
                    "RNA_weight"
                ]
            ),

        "second_weight":
            float(
                spec[
                    "ADT_weight"
                ]
            ),

        "embedding_dim":
            int(
                embedding.shape[1]
            ),

        "ARI":
            float(ari),

        "NMI":
            float(nmi),

        "NMI_average_method":
            "max",

        "runtime_sec":
            runtime_sec,

        "preprocessing":
            preprocessing_audit[
                "compatibility_preprocessing"
            ],

        "final_clustering":
            "PCA20+mclust_EEE_seed2020",
    }


    with open(
        metrics_path,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            result,
            f,
            indent=2,
        )


    print(
        "\n"
        + "-" * 110
    )

    print(
        "PRAGA RUN RESULT"
    )

    print(
        "-" * 110
    )

    print(
        f"Dataset     : {dataset_name}"
    )

    print(
        f"Seed        : {seed}"
    )

    print(
        f"Spots       : {spec['n_spots']}"
    )

    print(
        f"Target K    : {spec['K']}"
    )

    print(
        f"Predicted K : "
        f"{len(np.unique(pred))}"
    )

    print(
        f"Embedding   : "
        f"{embedding.shape}"
    )

    print(
        f"ARI         : "
        f"{ari:.12f}"
    )

    print(
        f"NMI         : "
        f"{nmi:.12f}"
    )

    print(
        f"Runtime     : "
        f"{runtime_sec:.2f} sec"
    )

    print(
        f"Evidence    : "
        f"{run_dir}"
    )


    # ========================================================
    # Cleanup
    # ========================================================

    del model
    del output
    del data
    del rna
    del second

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return result


print(
    "PASS: single-run PRAGA runner defined."
)

PASS: single-run PRAGA runner defined.


### Cell 10 — Run the remaining three seed0 PRAGA smoke tests

In [61]:
SMOKE_ROOT = (
    PRAGA_OUTPUT_ROOT
    / "smoke_final"
)

SMOKE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


smoke_results = []


for dataset_name in [
    "HLN-D1",
    "S2-E15",
    "S2-E18",
]:

    result = run_praga_once(
        dataset_name=
            dataset_name,

        seed=
            0,

        output_root=
            SMOKE_ROOT,

        overwrite=
            False,
    )

    smoke_results.append(
        result
    )


smoke_df = pd.DataFrame(
    smoke_results
)


display(
    smoke_df[
        [
            "dataset",
            "seed",
            "n_spots",
            "target_K",
            "predicted_K",
            "ARI",
            "NMI",
            "runtime_sec",
        ]
    ]
)


assert (
    smoke_df[
        "predicted_K"
    ]
    ==
    smoke_df[
        "target_K"
    ]
).all()


print(
    "\nPASS: remaining three smoke runs completed."
)


PRAGA RUN | HLN-D1 | seed=0


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/lib/python3.12/functools.py:912: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


HLN-D1: RNA feat = (3359, 30)
HLN-D1: ADT feat = (3359, 30)
HLN-D1: KNN_k=20, weights=[5, 5], init_k=11


  0%|          | 0/30 [00:00<?, ?it/s]/kaggle/working/PRAGA/PRAGA/Train_model.py:107: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/kaggle/working/PRAGA/PRAGA/Train_model.py:108: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
  3%|▎         | 1/30 [00:00<00:07,  4.08it/s]

tensor(30.5108, device='cuda:0', grad_fn=<AddBackward0>) tensor(27.3595, device='cuda:0', grad_fn=<DivBackward0>) 0.0
tensor(30.2576, device='cuda:0', grad_fn=<AddBackward0>) tensor(24.6144, device='cuda:0', grad_fn=<DivBackward0>) 0.0


...

tensor(12.0589, device='cuda:0', grad_fn=<AddBackward0>) tensor(0.0209, device='cuda:0', grad_fn=<DivBackward0>) tensor(-5.4639, device='cuda:0', grad_fn=<MulBackward0>)
tensor(12.0582, device='cuda:0', grad_fn=<AddBackward0>) tensor(0.0210, device='cuda:0', grad_fn=<DivBackward0>) tensor(-5.4656, device='cuda:0', grad_fn=<MulBackward0>)


100%|██████████| 300/300 [13:27<00:00,  2.69s/it]


tensor(12.0574, device='cuda:0', grad_fn=<AddBackward0>) tensor(0.0210, device='cuda:0', grad_fn=<DivBackward0>) tensor(-5.4648, device='cuda:0', grad_fn=<MulBackward0>)
Model training finished!

Infer time:  0.003326416015625
PCA20 shape : (2248, 20)
PCA20 finite: True
mclust K    : 16

--------------------------------------------------------------------------------------------------------------
PRAGA RUN RESULT
--------------------------------------------------------------------------------------------------------------
Dataset     : S2-E18
Seed        : 0
Spots       : 2248
Target K    : 16
Predicted K : 16
Embedding   : (2248, 64)
ARI         : 0.352017873965
NMI         : 0.475134628466
Runtime     : 828.90 sec
Evidence    : /kaggle/working/PRAGA_baseline/smoke_final/S2E18_seed0


,dataset,seed,n_spots,target_K,predicted_K,ARI,NMI,runtime_sec
0,HLN-D1,0,3359,11,11,0.193806,0.298272,23.602796
1,S2-E15,0,1939,15,15,0.440489,0.601843,764.021488
2,S2-E18,0,2248,16,16,0.352018,0.475135,828.896516



PASS: remaining three smoke runs completed.


### Cell 11 — Run 5 datasets × 10 seeds with resume

In [62]:
FORMAL_SEEDS = list(
    range(10)
)


print("=" * 110)
print("PRAGA FORMAL BENCHMARK")
print("=" * 110)

print(
    "Datasets:",
    DATASET_ORDER
)

print(
    "Seeds:",
    FORMAL_SEEDS
)

print(
    "Total expected runs:",
    len(DATASET_ORDER)
    *
    len(FORMAL_SEEDS)
)


all_results = []


for dataset_name in DATASET_ORDER:

    for seed in FORMAL_SEEDS:

        try:

            result = run_praga_once(
                dataset_name=
                    dataset_name,

                seed=
                    seed,

                output_root=
                    FORMAL_ROOT,

                overwrite=
                    False,
            )

            all_results.append(
                result
            )

        except Exception as e:

            print(
                "\n"
                + "!" * 110
            )

            print(
                f"FAILED: "
                f"{dataset_name} "
                f"seed={seed}"
            )

            print(
                repr(e)
            )

            print(
                "!" * 110
            )

            raise


formal_raw = pd.DataFrame(
    all_results
)


formal_raw = (
    formal_raw
    .sort_values(
        [
            "dataset",
            "seed",
        ]
    )
    .reset_index(
        drop=True
    )
)


RAW_CSV = (
    PRAGA_OUTPUT_ROOT
    / "PRAGA_5datasets_10seeds_RAW.csv"
)


formal_raw.to_csv(
    RAW_CSV,
    index=False,
)


print(
    "\nSaved:",
    RAW_CSV
)

print(
    "Completed runs:",
    len(formal_raw)
)


display(
    formal_raw[
        [
            "dataset",
            "seed",
            "ARI",
            "NMI",
            "runtime_sec",
        ]
    ]
)

PRAGA FORMAL BENCHMARK
Datasets: ['HLN-A1', 'HLN-D1', 'E18.5', 'S2-E15', 'S2-E18']
Seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Total expected runs: 50

PRAGA RUN | HLN-A1 | seed=0


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/lib/python3.12/functools.py:912: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


HLN-A1: RNA feat = (3484, 30)
HLN-A1: ADT feat = (3484, 30)
HLN-A1: KNN_k=20, weights=[5, 5], init_k=10


  0%|          | 0/30 [00:00<?, ?it/s]/kaggle/working/PRAGA/PRAGA/Train_model.py:107: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/kaggle/working/PRAGA/PRAGA/Train_model.py:108: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
  3%|▎         | 1/30 [00:00<00:06,  4.38it/s]

tensor(39.0317, device='cuda:0', grad_fn=<AddBackward0>) tensor(27.6541, device='cuda:0', grad_fn=<DivBackward0>) 0.0
tensor(38.1447, device='cuda:0', grad_fn=<AddBackward0>) tensor(24.8785, device='cuda:0', grad_fn=<DivBackward0>) 0.0


...

 99%|█████████▉| 298/300 [36:10<00:00,  8.20it/s]

tensor(11.9787, device='cuda:0', grad_fn=<AddBackward0>) tensor(0.0215, device='cuda:0', grad_fn=<DivBackward0>) tensor(-5.4589, device='cuda:0', grad_fn=<MulBackward0>)
tensor(11.9775, device='cuda:0', grad_fn=<AddBackward0>) tensor(0.0211, device='cuda:0', grad_fn=<DivBackward0>) tensor(-5.4629, device='cuda:0', grad_fn=<MulBackward0>)


100%|██████████| 300/300 [36:11<00:00,  7.24s/it]


tensor(11.9762, device='cuda:0', grad_fn=<AddBackward0>) tensor(0.0211, device='cuda:0', grad_fn=<DivBackward0>) tensor(-5.4652, device='cuda:0', grad_fn=<MulBackward0>)
Model training finished!

Infer time:  0.00235748291015625
PCA20 shape : (2248, 20)
PCA20 finite: True
mclust K    : 16

--------------------------------------------------------------------------------------------------------------
PRAGA RUN RESULT
--------------------------------------------------------------------------------------------------------------
Dataset     : S2-E18
Seed        : 9
Spots       : 2248
Target K    : 16
Predicted K : 16
Embedding   : (2248, 64)
ARI         : 0.326071958160
NMI         : 0.492151398633
Runtime     : 2191.54 sec
Evidence    : /kaggle/working/PRAGA_baseline/formal/S2E18_seed9

Saved: /kaggle/working/PRAGA_baseline/PRAGA_5datasets_10seeds_RAW.csv
Completed runs: 50


,dataset,seed,ARI,NMI,runtime_sec
0,E18.5,0,0.619755,0.620758,674.398380
1,E18.5,1,0.527284,0.589601,1002.814961
2,E18.5,2,0.527942,0.577047,740.726273
3,E18.5,3,0.519993,0.590708,784.296499
4,E18.5,4,0.597059,0.609564,1279.134429
5,E18.5,5,0.580906,0.601505,946.881583
6,E18.5,6,0.460449,0.569380,497.073450
7,E18.5,7,0.507632,0.579141,667.820817
8,E18.5,8,0.603200,0.608395,585.161538
9,E18.5,9,0.547171,0.582059,1837.788869


### Cell 12 — Summarize and validate all 50 formal PRAGA runs

In [63]:
# ============================================================
# Reload every completed run from disk
# ============================================================

rows = []


for dataset_name in DATASET_ORDER:

    tag = dataset_tag(
        dataset_name
    )

    for seed in FORMAL_SEEDS:

        run_dir = (
            FORMAL_ROOT
            / f"{tag}_seed{seed}"
        )

        metrics_path = (
            run_dir
            / "metrics.json"
        )

        assert metrics_path.exists(), (
            f"Missing: {metrics_path}"
        )

        with open(
            metrics_path,
            "r",
            encoding="utf-8",
        ) as f:

            row = json.load(
                f
            )

        rows.append(
            row
        )


raw = pd.DataFrame(
    rows
)


raw = (
    raw
    .sort_values(
        [
            "dataset",
            "seed",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 50-run validation
# ============================================================

assert (
    len(raw)
    ==
    50
)


for dataset_name in DATASET_ORDER:

    sub = raw[
        raw[
            "dataset"
        ]
        ==
        dataset_name
    ]

    assert (
        len(sub)
        ==
        10
    )

    assert (
        sorted(
            sub[
                "seed"
            ].tolist()
        )
        ==
        list(
            range(10)
        )
    )

    spec = (
        DATASET_SPECS[
            dataset_name
        ]
    )

    assert (
        sub[
            "n_spots"
        ]
        ==
        spec[
            "n_spots"
        ]
    ).all()

    assert (
        sub[
            "target_K"
        ]
        ==
        spec[
            "K"
        ]
    ).all()

    assert (
        sub[
            "predicted_K"
        ]
        ==
        spec[
            "K"
        ]
    ).all()

    assert (
        sub[
            "training_init_K"
        ]
        ==
        spec[
            "init_k"
        ]
    ).all()

    assert (
        sub[
            "epochs"
        ]
        ==
        spec[
            "epochs"
        ]
    ).all()


assert np.isfinite(
    raw[
        [
            "ARI",
            "NMI",
            "runtime_sec",
        ]
    ].to_numpy()
).all()


# ============================================================
# Summary
# ============================================================

summary_rows = []


for dataset_name in DATASET_ORDER:

    sub = raw[
        raw[
            "dataset"
        ]
        ==
        dataset_name
    ]


    summary_rows.append(
        {
            "dataset":
                dataset_name,

            "n_runs":
                len(sub),

            "ARI_mean":
                sub[
                    "ARI"
                ].mean(),

            "ARI_std":
                sub[
                    "ARI"
                ].std(
                    ddof=0
                ),

            "NMI_mean":
                sub[
                    "NMI"
                ].mean(),

            "NMI_std":
                sub[
                    "NMI"
                ].std(
                    ddof=0
                ),

            "runtime_mean_sec":
                sub[
                    "runtime_sec"
                ].mean(),

            "runtime_std_sec":
                sub[
                    "runtime_sec"
                ].std(
                    ddof=0
                ),
        }
    )


summary = pd.DataFrame(
    summary_rows
)


macro_ari = float(
    summary[
        "ARI_mean"
    ].mean()
)

macro_nmi = float(
    summary[
        "NMI_mean"
    ].mean()
)


RAW_CSV = (
    PRAGA_OUTPUT_ROOT
    / "PRAGA_5datasets_10seeds_RAW.csv"
)

SUMMARY_CSV = (
    PRAGA_OUTPUT_ROOT
    / "PRAGA_5datasets_10seeds_SUMMARY.csv"
)


raw.to_csv(
    RAW_CSV,
    index=False,
)

summary.to_csv(
    SUMMARY_CSV,
    index=False,
)


protocol = {
    "method":
        "PRAGA",

    "datasets":
        DATASET_ORDER,

    "seeds":
        FORMAL_SEEDS,

    "n_formal_runs":
        50,

    "RNA_ADT": {
        "KNN_k": 20,
        "RNA_weight": 5,
        "ADT_weight": 5,
        "epochs": 30,
    },

    "RNA_ATAC": {
        "KNN_k": 20,
        "RNA_weight": 1,
        "ATAC_weight": 10,
        "epochs": 300,
        "compatibility_preprocessing":
            "remove zero-total peaks; "
            "PRAGA LSI; "
            "column-wise zero-mean / "
            "unit sample-std scaling",
        "spot_filtering":
            False,
    },

    "init_k": {
        name:
            int(
                DATASET_SPECS[
                    name
                ][
                    "init_k"
                ]
            )
        for name
        in DATASET_ORDER
    },

    "clustering":
        "PRAGA embedding -> "
        "PCA20 -> mclust EEE",

    "mclust_seed":
        2020,

    "NMI_average_method":
        "max",

    "summary_std_ddof":
        0,

    "macro_ARI":
        macro_ari,

    "macro_NMI":
        macro_nmi,
}


PROTOCOL_JSON = (
    PRAGA_OUTPUT_ROOT
    / "PRAGA_PROTOCOL.json"
)


with open(
    PROTOCOL_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        protocol,
        f,
        indent=2,
    )


print(
    "=" * 110
)

print(
    "PRAGA 50-RUN VALIDATION PASS"
)

print(
    "=" * 110
)


display(
    summary
)


print(
    f"\nMacro ARI = "
    f"{macro_ari:.6f}"
)

print(
    f"Macro NMI = "
    f"{macro_nmi:.6f}"
)

print(
    "\nRAW     :",
    RAW_CSV
)

print(
    "SUMMARY :",
    SUMMARY_CSV
)

print(
    "PROTOCOL:",
    PROTOCOL_JSON
)

PRAGA 50-RUN VALIDATION PASS


,dataset,n_runs,ARI_mean,ARI_std,NMI_mean,NMI_std,runtime_mean_sec,runtime_std_sec
0,HLN-A1,10,0.257314,0.015069,0.340109,0.014839,26.058449,1.429496
1,HLN-D1,10,0.189588,0.013389,0.301240,0.005527,25.396493,1.173183
2,E18.5,10,0.549139,0.047522,0.592816,0.015795,901.609680,379.600515
3,S2-E15,10,0.447151,0.037155,0.583931,0.022677,918.337649,267.645452
4,S2-E18,10,0.384596,0.056645,0.474014,0.016776,1215.254894,382.865201



Macro ARI = 0.365558
Macro NMI = 0.458422

RAW     : /kaggle/working/PRAGA_baseline/PRAGA_5datasets_10seeds_RAW.csv
SUMMARY : /kaggle/working/PRAGA_baseline/PRAGA_5datasets_10seeds_SUMMARY.csv
PROTOCOL: /kaggle/working/PRAGA_baseline/PRAGA_PROTOCOL.json


### Cell 13 — Freeze all completed PRAGA formal results### Cell 13 — Freeze all completed PRAGA formal results

In [64]:
from pathlib import Path
from datetime import datetime
import json
import shutil
import platform
import sys
import numpy as np
import pandas as pd
import torch
import sklearn
import scanpy as sc
import anndata


ARCHIVE_ROOT = (
    Path("/kaggle/working")
    / "PRAGA_FINAL_ARCHIVE"
)


if ARCHIVE_ROOT.exists():

    shutil.rmtree(
        ARCHIVE_ROOT
    )


ARCHIVE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# Copy top-level result files
# ============================================================

top_level_files = [
    PRAGA_OUTPUT_ROOT
    / "PRAGA_5datasets_10seeds_RAW.csv",

    PRAGA_OUTPUT_ROOT
    / "PRAGA_5datasets_10seeds_SUMMARY.csv",

    PRAGA_OUTPUT_ROOT
    / "PRAGA_PROTOCOL.json",
]


for src in top_level_files:

    if src.exists():

        shutil.copy2(
            src,
            ARCHIVE_ROOT
            / src.name,
        )


# ============================================================
# Copy all 50 formal run folders
# ============================================================

formal_archive = (
    ARCHIVE_ROOT
    / "formal_runs"
)

formal_archive.mkdir(
    parents=True,
    exist_ok=True,
)


completed_runs = []
missing_runs = []


required_files = [
    "embedding.npy",
    "pred_labels.npy",
    "gt_labels.npy",
    "coords.npy",
    "spot_ids.npy",
    "metrics.json",
    "preprocessing_audit.json",
]


for dataset_name in DATASET_ORDER:

    tag = dataset_tag(
        dataset_name
    )

    for seed in range(10):

        run_name = (
            f"{tag}_seed{seed}"
        )

        src_dir = (
            FORMAL_ROOT
            / run_name
        )

        dst_dir = (
            formal_archive
            / run_name
        )


        missing = [
            name
            for name in required_files
            if not (
                src_dir
                / name
            ).exists()
        ]


        if src_dir.exists():

            shutil.copytree(
                src_dir,
                dst_dir,
                dirs_exist_ok=True,
            )


        if len(missing) == 0:

            completed_runs.append(
                {
                    "dataset":
                        dataset_name,

                    "seed":
                        int(seed),

                    "run_name":
                        run_name,
                }
            )

        else:

            missing_runs.append(
                {
                    "dataset":
                        dataset_name,

                    "seed":
                        int(seed),

                    "run_name":
                        run_name,

                    "missing_files":
                        missing,
                }
            )


# ============================================================
# Save environment / protocol snapshot
# ============================================================

manifest = {
    "method":
        "PRAGA",

    "archive_time":
        datetime.now()
        .astimezone()
        .isoformat(),

    "datasets":
        list(
            DATASET_ORDER
        ),

    "formal_seeds":
        list(
            range(10)
        ),

    "expected_runs":
        50,

    "completed_runs":
        len(
            completed_runs
        ),

    "missing_runs":
        len(
            missing_runs
        ),

    "PRAGA_git_commit":
        str(
            PRAGA_COMMIT
        ),

    "result_policy":
        (
            "All completed formal runs are archived "
            "regardless of ARI/NMI performance."
        ),

    "RNA_ADT": {
        "epochs": 30,
        "KNN_k": 20,
        "RNA_weight": 5,
        "ADT_weight": 5,
    },

    "RNA_ATAC": {
        "epochs": 300,
        "KNN_k": 20,
        "RNA_weight": 1,
        "ATAC_weight": 10,
        "compatibility_preprocessing":
            (
                "remove zero-total peaks; "
                "PRAGA LSI; "
                "component-wise standardization"
            ),
        "spot_filtering":
            False,
    },

    "final_clustering":
        "PRAGA embedding -> PCA20 -> mclust EEE",

    "mclust_seed":
        2020,

    "NMI_average_method":
        "max",

    "summary_std_ddof":
        0,

    "environment": {
        "python":
            sys.version.split()[0],

        "torch":
            torch.__version__,

        "numpy":
            np.__version__,

        "sklearn":
            sklearn.__version__,

        "scanpy":
            sc.__version__,

        "anndata":
            anndata.__version__,

        "gpu":
            (
                torch.cuda.get_device_name(0)
                if torch.cuda.is_available()
                else None
            ),
    },
}


with open(
    ARCHIVE_ROOT
    / "ARCHIVE_MANIFEST.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
    )


with open(
    ARCHIVE_ROOT
    / "MISSING_RUNS.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        missing_runs,
        f,
        indent=2,
    )


print("=" * 100)
print("PRAGA FINAL ARCHIVE CHECK")
print("=" * 100)

print(
    "Expected runs :",
    50
)

print(
    "Completed     :",
    len(
        completed_runs
    )
)

print(
    "Missing       :",
    len(
        missing_runs
    )
)

print(
    "Archive root  :",
    ARCHIVE_ROOT
)


if len(missing_runs) == 0:

    print(
        "\nPASS: 50/50 formal runs archived."
    )

else:

    print(
        "\nWARNING: some formal runs are incomplete."
    )

    display(
        pd.DataFrame(
            missing_runs
        )
    )

PRAGA FINAL ARCHIVE CHECK
Expected runs : 50
Completed     : 50
Missing       : 0
Archive root  : /kaggle/working/PRAGA_FINAL_ARCHIVE

PASS: 50/50 formal runs archived.


### Cell 14 — Create the final PRAGA result ZIP### Cell 14 — Create the final PRAGA result ZIP

In [65]:
import shutil
from pathlib import Path


ZIP_BASE = (
    Path("/kaggle/working")
    / "PRAGA_FINAL_5datasets_10seeds"
)


zip_path = shutil.make_archive(
    base_name=str(
        ZIP_BASE
    ),
    format="zip",
    root_dir=ARCHIVE_ROOT.parent,
    base_dir=ARCHIVE_ROOT.name,
)


zip_path = Path(
    zip_path
)


print("=" * 100)
print("FINAL PRAGA RESULT PACKAGE")
print("=" * 100)

print(
    "ZIP file:",
    zip_path
)

print(
    "Size MB :",
    f"{zip_path.stat().st_size / 1024 / 1024:.2f}"
)


summary_path = (
    ARCHIVE_ROOT
    / "PRAGA_5datasets_10seeds_SUMMARY.csv"
)


if summary_path.exists():

    summary = pd.read_csv(
        summary_path
    )

    print(
        "\nFrozen summary:"
    )

    display(
        summary
    )


print(
    "\nPASS: final PRAGA results packaged."
)

FINAL PRAGA RESULT PACKAGE
ZIP file: /kaggle/working/PRAGA_FINAL_5datasets_10seeds.zip
Size MB : 35.93

Frozen summary:


,dataset,n_runs,ARI_mean,ARI_std,NMI_mean,NMI_std,runtime_mean_sec,runtime_std_sec
0,HLN-A1,10,0.257314,0.015069,0.340109,0.014839,26.058449,1.429496
1,HLN-D1,10,0.189588,0.013389,0.301240,0.005527,25.396493,1.173183
2,E18.5,10,0.549139,0.047522,0.592816,0.015795,901.609680,379.600515
3,S2-E15,10,0.447151,0.037155,0.583931,0.022677,918.337649,267.645452
4,S2-E18,10,0.384596,0.056645,0.474014,0.016776,1215.254894,382.865201



PASS: final PRAGA results packaged.
